# Week 8 (Tue) - Class Activities in CS 315

You are given the file `netflix_top10_countries.tsv` in the folder data, solve the following tasks using it.

**Table of Contents**

1. [Q1: Load the file](#q1)
2. [Q2: Show number of rows](#q2)
3. [Q3: Show column names](#q3)
4. [Q4: Show number of total countries](#q4)
5. [Q5: Show number of TV series](#q5)
6. [Q6: Number of TV series in South Korea](#q6)
7. [Q7: Total weeks of Squid Games in Italy ](#q7)
8. [Q8: Time frame for a given country](#q8)
9. [Q9: Is number of weeks in ranking equal?](#q9)
10. [Q10: Country with fewest weeks](#q10)


In [35]:
import pandas as pd

<a id="q1"></a>
### Q1: Load the file

In [36]:
df = pd.read_csv("netflix_top10_countries.tsv", sep="\t")
df.head(1)

,country_name,country_iso2,week,category,weekly_rank,show_title,season_title,cumulative_weeks_in_top_10
0,Argentina,AR,2026-10-04,Films,1,UNABOMBER,NaN,2


<a id="q2"></a>
### Q2: Show number of rows

In [37]:
df.shape[0]

512200

<a id="q3"></a>
### Q3: Show column names

In [38]:
df.columns

Index(['country_name', 'country_iso2', 'week', 'category', 'weekly_rank',
       'show_title', 'season_title', 'cumulative_weeks_in_top_10'],
      dtype='object')

<a id="q4"></a>
### Q4: Show number of total countries

In [39]:
df["country_name"].nunique()

94

<a id="q5"></a>
### Q5: Show number of TV series

In [40]:
df["show_title"].nunique()

12197

<a id="q6"></a>
### Q6: Number of TV series in South Korea

In [41]:
sk_df = df[df["country_name"] == "South Korea"].copy()
sk_df["show_title"].nunique()

1568

<a id="q7"></a>
### Q7: Total weeks of Squid Game in Italy

In [42]:
it_df = df[df["country_name"] == "Italy"].copy()
it_sg_df = it_df[it_df["show_title"] == "Squid Game"].copy()
it_sg_df["cumulative_weeks_in_top_10"].max()

np.int64(21)

<a id="q8"></a>
### Q8: Time frame for a given country

Write a function that given a country name prints out the start and end date of their rankings.

Once you have created the function, call it for a few countries to test it.

In [43]:
def show_timeframe(df, country):
    country_df = df[df["country_name"] == country].copy()
    min_date = country_df["week"].min()
    max_date = country_df["week"].max()
    print(f"Netflix rankings for {country} exist between {min_date} and {max_date}")

In [44]:
show_timeframe(df, "Chile")

Netflix rankings for Chile exist between 2021-07-04 and 2026-10-04


In [45]:
show_timeframe(df, "Russia")

Netflix rankings for Russia exist between 2021-07-04 and 2022-02-27


<a id="q9"></a>
### Q9: Is number of weeks equal for each country?

The total number of weeks in the rankings is `len(df['week'].unique())`.

The total number of countries is `len(df['country_name'].unique())`.  

For every week, each country has 20 rows.   

If they all stayed the same number of weeks, then the product of these three weeks would be the same as the total number of rows.  

In [46]:
num_weeks = len(df["week"].unique())
num_countries = len(df["country_name"].unique())
print(f"Expected number of rows: {num_weeks * num_countries * 20}")
print(f"Actual number of rows: {df.shape[0]}")
print(f"Are they the same? {(num_weeks * num_countries * 20) == df.shape[0]}")

Expected number of rows: 517000
Actual number of rows: 512200
Are they the same? False


We should have expected this to be `False` since we saw in Q8 that Russia stopped being in the rankings after it attacked Ukraine in February 2022.

<a id="q10"></a>
### Q10: Country with fewest weeks

Write a one-liner that will show the country with the fewest weeks in the Netflix rankings

In [47]:
df["country_name"].value_counts().sort_values(ascending=True).head(1)

country_name
Russia    700
Name: count, dtype: int64

## Bonus task: rank2score

In [48]:
def rank2score(rank):
    rank_dict = {
        1:25, 2:18, 3:15, 4:12, 5:10,
        6:8, 7:6, 8:4, 9:2, 10:1
    }
    if rank in rank_dict.keys():
        return rank_dict[rank]
    else:
        print("Rank input must be an integer value 1-10.")
        return None

In [49]:
df = df.drop(columns=["country_iso2", "category", "season_title"])

In [51]:
df["score"] = df["weekly_rank"].apply(rank2score)
df = df[["country_name", "week", "weekly_rank", "score", "show_title", "cumulative_weeks_in_top_10"]]

In [52]:
df.head(10)

,country_name,week,weekly_rank,score,show_title,cumulative_weeks_in_top_10
0,Argentina,2026-10-04,1,25,UNABOMBER,2
1,Argentina,2026-10-04,2,18,The Equalizer 3,1
2,Argentina,2026-10-04,3,15,The Substance,1
3,Argentina,2026-10-04,4,12,Demon Slayer: Kimetsu no Yaiba Infinity Castle I,1
4,Argentina,2026-10-04,5,10,What a Father!,1
5,Argentina,2026-10-04,6,8,The Widower: 'Til Death Do Us Part,1
6,Argentina,2026-10-04,7,6,Blood Work,2
7,Argentina,2026-10-04,8,4,Doing Life,1
8,Argentina,2026-10-04,9,2,The Mummy,1
9,Argentina,2026-10-04,10,1,Black Adam,2


In [128]:
def get_show_total_score(show_title):
    show_df = df[df["show_title"] == show_title]
    new_df = pd.DataFrame(show_df["country_name"].unique(), columns=["country_name"])

    total_scores = show_df.groupby("country_name")["score"].sum().reset_index()
    total_weeks = pd.DataFrame(show_df["country_name"].value_counts()).reset_index()

    new_df = new_df.merge(total_scores, on="country_name", how="left")
    new_df = new_df.merge(total_weeks, on="country_name", how="left")
    new_df = new_df.rename(columns={"score": "total_score", "count":"total_weeks"})

    rankings_lists = {}
    for country in new_df["country_name"].unique():
        show_country_df = show_df[show_df["country_name"] == country]
        rankings = show_country_df["weekly_rank"].tolist()
        rankings_lists[country] = rankings

    new_df["rankings_list"] = new_df["country_name"].map(rankings_lists)

    return new_df.sort_values(by=["total_score"], ascending=False)

In [129]:
get_show_total_score("Lovely Runner")

,country_name,total_score,total_weeks,rankings_list
12,Vietnam,75,7,"[8, 7, 6, 5, 4, 1, 5]"
2,Indonesia,69,7,"[9, 7, 6, 7, 4, 1, 5]"
3,Malaysia,62,6,"[9, 9, 7, 3, 1, 4]"
6,Philippines,41,4,"[7, 5, 3, 5]"
4,Maldives,32,4,"[10, 6, 3, 6]"
9,Sri Lanka,32,4,"[10, 7, 3, 5]"
0,Bangladesh,30,3,"[5, 4, 6]"
1,India,26,3,"[8, 4, 5]"
7,Singapore,20,4,"[9, 7, 5, 9]"
11,Thailand,20,3,"[7, 8, 5]"


In [130]:
get_show_total_score("Squid Game")

,country_name,total_score,total_weeks,rankings_list
33,India,816,69,"[9, 9, 4, 4, 10, 3, 7, 9, 2, 5, 6, 1, 4, 5, 1,..."
61,Pakistan,808,66,"[10, 9, 6, 9, 3, 6, 7, 2, 5, 7, 1, 4, 5, 1, 3,..."
5,Bangladesh,807,66,"[9, 8, 8, 5, 4, 7, 1, 5, 9, 1, 4, 6, 1, 3, 4, ..."
40,Jordan,772,50,"[9, 8, 4, 6, 4, 3, 7, 1, 6, 7, 1, 2, 3, 1, 2, ..."
7,Bolivia,760,52,"[10, 7, 6, 2, 8, 2, 6, 9, 1, 4, 5, 1, 2, 3, 1,..."
...,...,...,...,...
27,Guadeloupe,307,21,"[10, 5, 1, 1, 8, 1, 9, 3, 1, 6, 1, 5, 10, 7, 6..."
35,Ireland,302,20,"[9, 5, 2, 1, 1, 5, 6, 3, 10, 2, 6, 1, 10, 5, 3..."
50,Martinique,298,20,"[5, 1, 1, 8, 1, 9, 3, 1, 6, 1, 5, 10, 9, 7, 5,..."
3,Bahamas,285,18,"[2, 1, 1, 5, 3, 1, 5, 1, 7, 10, 5, 4, 4, 3, 2,..."
